# EDA - Mapa de Estaciones Meteorológicas (Agrocabildo)
Pintar todas las estaciones y resaltar cuáles tienen datos desde el 01-01-2022 para entrenar la IA.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [ ]:
# Cargar metadatos de estaciones y el histórico de clima
estaciones = pd.read_parquet('../data/estaciones_agrocabildo.parquet')
clima = pd.read_parquet('../data/clima_horario_agrocabildo.parquet')

In [ ]:
# Convertir a Geodataframe para poder pintar en mapa
geometry = [Point(xy) for xy in zip(estaciones['longitud'], estaciones['latitud'])]
gdf_est = gpd.GeoDataFrame(estaciones, geometry=geometry, crs='EPSG:4326')

In [ ]:
# 1. Identificar estaciones con datos posteriores a 2022-01-01
# (el CSV original suele traer la fecha como texto, la convertimos)
clima['fecha_hora'] = pd.to_datetime(clima['fecha_hora'])
clima_reciente = clima[clima['fecha_hora'] >= '2022-01-01']

# Sacamos los IDs de las estaciones que sí reportaron en esta época
estaciones_validas = clima_reciente['estacion'].unique()

# Marcamos en el dataframe principal cuáles son válidas
gdf_est['es_valida_2022'] = gdf_est['estacion'].isin(estaciones_validas)

print(f"Total de estaciones en la isla: {len(gdf_est)}")
print(f"Estaciones que sobreviven al filtro (datos >= 2022): {gdf_est['es_valida_2022'].sum()}")

In [ ]:
# 2. Visualizar en el mapa
fig, ax = plt.subplots(figsize=(12, 10))

# Pintar las descartadas en gris con una X
gdf_est[gdf_est['es_valida_2022'] == False].plot(
    ax=ax, color='gray', marker='x', markersize=80, label='Descartadas (< 2022)'
)

# Pintar las válidas en verde brillante
gdf_est[gdf_est['es_valida_2022'] == True].plot(
    ax=ax, color='#00ff00', edgecolor='black', marker='o', markersize=100, label='Válidas (>= 2022)'
)

plt.title('Red de Estaciones de Agrocabildo\nFiltro de Calidad: Datos desde el 01-01-2022')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)

# Añadir etiquetas con los nombres de las estaciones válidas para ver cuáles son
for idx, row in gdf_est[gdf_est['es_valida_2022'] == True].iterrows():
    ax.annotate(row['nombre'], (row.geometry.x, row.geometry.y), xytext=(5, 5), 
                textcoords='offset points', fontsize=8)

plt.show()